# Chạy Thực Nghiệm Đồ Án — Phần 2 (CNTK-Nystrom + 5 thực nghiệm bổ sung)

Notebook này là **phần tách ra** từ `all-experiments.ipynb`: bắt đầu từ mục
"Tái hiện đúng thực nghiệm gốc trên CIFAR-10 thật (bài báo gốc)" trở về sau
(pipeline CNTK 6 lớp + Nystrom trên CIFAR-10 và FashionMNIST, cùng 5 thực
nghiệm bổ sung: GMM, Dictionary Selection, Batch Active Learning, Deep
network coresets, Joint coresets).

**Lý do tách:** notebook gốc chạy tổng cộng quá 12 giờ (giới hạn một phiên
Kaggle) nếu chạy hết Continual Learning + Streaming + phần CNTK-Nystrom +
5 thực nghiệm bổ sung trong cùng một notebook. Notebook này chỉ chứa phần
**CNTK-Nystrom trở về sau**; phần Continual Learning (Table 3) + Streaming
(Table 5/6) + kiểm chứng chuẩn hóa vẫn ở `all-experiments.ipynb` như cũ.

**Thứ tự chạy trong notebook này: nhẹ → nặng.** GMM và Dictionary Selection
(rẻ, vài phút) chạy trước, Batch Active Learning (audio, ~30-60 phút) ở
giữa, rồi tới CNTK-Nystrom trên CIFAR-10 và FashionMNIST (nặng — tính
Nystrom feature cho toàn bộ ảnh train qua CNTK 6 lớp, có thể mất hàng giờ),
và cuối cùng là 2 thực nghiệm dùng mạng sâu (WideResNet/VGG16/MobileNetV2 —
nặng nhất, mỗi lần chọn coreset đều phải huấn luyện lại mạng). Nếu phiên
Kaggle bị ngắt giữa chừng, các phần rẻ hơn ở đầu vẫn kịp có kết quả.

**HƯỚNG DẪN:**
1. Bật GPU (T4 x2 hoặc P100) trong Settings của Kaggle.
2. Chạy Ô Số 1 để cài đặt môi trường + tải mã nguồn.
3. Bấm **Restart Session / Restart Kernel** khi Kaggle yêu cầu.
4. Chạy tiếp Ô vá JAX + Ô kiểm tra GPU, rồi chạy các phần bên dưới từ trên
   xuống (mỗi phần đều có smoke test trước, chạy đầy đủ sau).


In [1]:
# Ô SỐ 1: CÀI ĐẶT MÔI TRƯỜNG VÀ TẢI MÃ NGUỒN
!git clone https://github.com/quachthanhhmd/bilevel-coresets.git
%cd bilevel-coresets

Cloning into 'bilevel-coresets'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 328 (delta 161), reused 310 (delta 143), pack-reused 0 (from 0)
Receiving objects: 100% (328/328), 651.16 KiB | 5.38 MiB/s, done.
Resolving deltas: 100% (161/161), done.
/kaggle/working/bilevel-coresets


In [2]:
!git checkout bugfix/fixing-jax-error

Branch 'bugfix/fixing-jax-error' set up to track remote branch 'bugfix/fixing-jax-error' from 'origin'.
Switched to a new branch 'bugfix/fixing-jax-error'


⚠️ **CẢNH BÁO QUAN TRỌNG:** Dừng lại tại đây! Bạn phải bấm `Restart Session` (hoặc `Restart Kernel`) trước khi chạy ô tiếp theo.

In [3]:
# 1. Hạ cấp setuptools để vá lỗi môi trường build của Python 3.12
!pip install "setuptools<70.0.0"

# 2. Cài duy nhất thư viện mô phỏng mạng CNN còn thiếu
!pip install neural-tangents

!pip install --upgrade jax jaxlib==0.1.56+cuda101 -f https://storage.googleapis.com/jax-releases/jax_releases.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 13.1 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.7/248.7 kB 5.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 6.2 MB/s eta 0:00:00
Looking in links: https://storage.googleapis.com/jax-releases/jax_releases.html
ERROR: Ignored the following yanked versions: 0.4.32
ERROR: Could not find a version that satisfies the requirement jaxlib==0.1.56+cuda101 (from versions: 0.4.17, 0.4.18, 0.4.19, 0.4.20, 0.4.21, 0.4.22, 0.4.23, 0.4.24, 0.4.25, 0.4.26, 0.4.27, 0.4.28, 0.4.29, 0.4.30, 0.4.31, 0.4.32, 0.4.33, 0.4.34, 0.4.35, 0.4.36, 0.4.38, 0.5.0, 0.5.1, 0.5.3, 0.6.0, 0.6.1, 0.6.2, 0.7.0, 0.7.1, 0.7.2, 0.8.0, 0.8.1, 0.8.2, 0.8.3, 0.9.0, 0.9.0.1, 0.9.1, 0.9.2, 0.10.0, 0.10.1, 0.10.2, 0.11.0)
ERROR: No matching di

In [4]:
import jax
import jax.core
import jax._src.core
import jax.tree_util
import jax.util

# 1. Vá các class lõi bị giấu (Đã thêm Primitive)
missing_classes = [
    'Jaxpr', 'JaxprEqn', 'Literal', 'Var', 'DropVar', 
    'ClosedJaxpr', 'ShapedArray', 'Value', 'MainTrace', 'Trace',
    'Primitive'  # <--- CHÍNH LÀ THỦ PHẠM MỚI NHẤT
]
for cls_name in missing_classes:
    if hasattr(jax._src.core, cls_name):
        setattr(jax.core, cls_name, getattr(jax._src.core, cls_name))

# 2. Vá hàm tree_multimap
if not hasattr(jax.tree_util, 'tree_multimap'):
    jax.tree_util.tree_multimap = jax.tree_util.tree_map

# 3. Vá hàm safe_map và safe_zip cho jax.util
def custom_safe_map(f, *args):
    return list(map(f, *args))

def custom_safe_zip(*args):
    return list(zip(*args))

jax.util.safe_map = custom_safe_map
jax.util.safe_zip = custom_safe_zip

print("Đã vá nóng JAX V4: Bổ sung Primitive! Sẵn sàng nạp Neural Tangents.")

Đã vá nóng JAX V4: Bổ sung Primitive! Sẵn sàng nạp Neural Tangents.


### Kiểm tra GPU trước khi chạy các thực nghiệm nặng bên dưới

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "KHONG CO GPU! Vao Settings > Accelerator, chon GPU (T4 x2 hoac P100), "
        "bam Save, roi chay lai notebook tu dau. Cac script sau se crash hoac cuc ky "
        "cham neu chay tren CPU (CNTK 6 lop, WideResNet)."
    )
print('GPU OK:', torch.cuda.get_device_name(0))

# Dam bao dang o dung thu muc repo (can thiet sau khi Restart Session, cwd bi reset).
%cd /kaggle/working/bilevel-coresets


### Coreset cho Gaussian Mixture Model

Dữ liệu tổng hợp 2 chiều, GMM 5 thành phần, forward selection từng điểm một (bắt đầu từ 10 điểm ngẫu nhiên), IHVP qua conjugate gradient. So sánh Uniform / Sensitivity Coreset / Bilevel Coreset. Rẻ -- chạy vài chục giây trên CPU, không cần GPU.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_gmm_paper.py',
                '--output-dir', 'experiments'], check=True)


In [ ]:
from IPython.display import Image, display
for img_name in ['gmm_relative_nll_error.png', 'gmm_contours.png']:
    print(f'--- {img_name} ---')
    display(Image(filename=f'experiments/{img_name}'))


### Dictionary Selection cho Compressed Sensing

3 panel: vector thưa tổng hợp (dictionary Gaussian ngẫu nhiên), MNIST với dictionary wavelet db1, MNIST với VAE (generative-model recovery, Bora et al. 2017). λ=0.01 xuyên suốt, đúng paper. Cần tải MNIST -- nếu môi trường chặn tải, bỏ `mnist_wavelet,mnist_vae` khỏi `--panels` và chỉ chạy `synthetic`.

Chạy Ô smoke-test bên dưới trước (quy mô rất nhỏ) để chắc pipeline hoạt động, rồi mới chạy Ô đầy đủ.

In [ ]:
import subprocess
# Smoke test: quy mô nhỏ, không phải tham số mặc định của script.
subprocess.run(['python', 'experiments/run_dictionary_selection_paper.py',
                '--synth-n', '64', '--synth-d', '32', '--synth-dict-size', '128',
                '--synth-sizes', '4,16', '--mnist-n', '30', '--mnist-sizes', '4,16',
                '--vae-epochs', '2', '--vae-train-n', '500',
                '--output-dir', 'experiments/dictionary_smoketest'], check=True)
print('Smoke test OK -- co the chay O day du ben duoi.')


**Chạy đầy đủ (tham số mặc định = đúng paper):** 1024 vector 128 chiều, dictionary 16384 phần tử cho panel synthetic; 250 ảnh MNIST cho 2 panel còn lại.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_dictionary_selection_paper.py',
                '--output-dir', 'experiments'], check=True)


In [ ]:
from IPython.display import Image, display
for img_name in ['dictionary_synthetic_l2.png', 'dictionary_synthetic_l1.png',
                 'dictionary_mnist_wavelet_l2.png', 'dictionary_mnist_wavelet_l1.png',
                 'dictionary_mnist_vae_gm.png']:
    import os
    path = f'experiments/{img_name}'
    if os.path.exists(path):
        print(f'--- {img_name} ---')
        display(Image(filename=path))


### Batch Active Learning bán giám sát (audio)

Spoken Digit / Speech Commands V2, mel-spectrogram 32x32, WideResNet-28-10, MixMatch. So sánh Uniform / Max-entropy / K-center / BADGE / BiCo (proxy CNTK-Nystrom). Cần `torchaudio` (chưa cài ở Ô số 1) và tải dữ liệu audio lần đầu (Spoken Digit qua git clone, Speech Commands qua torchaudio).

⚠️ **Nặng:** mỗi vòng chọn mẫu đều huấn luyện lại MixMatch từ đầu; paper dùng 6 seed × 5 phương pháp. Chạy smoke-test trước, sau đó cân nhắc giảm `--seeds` nếu thời gian hạn chế.

In [ ]:
!pip install torchaudio


In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_batch_active_learning_paper.py',
                '--dataset', 'spoken_digit', '--smoke-test',
                '--output-dir', 'experiments/active_learning_smoketest'], check=True)
print('Smoke test OK -- co the chay O day du ben duoi.')


**Chạy đầy đủ (Spoken Digit, tham số paper):** start=10, batch=10, budget=200, 6 seed. Có thể giảm `--seeds` (vd. 3) nếu lo về thời gian.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_batch_active_learning_paper.py',
                '--dataset', 'spoken_digit', '--seeds', '6',
                '--output-dir', 'experiments'], check=True)


In [ ]:
from IPython.display import Image, display
display(Image(filename='experiments/batch_active_learning_spoken_digit.png'))


## CNTK-Nystrom trên CIFAR-10 và FashionMNIST (nặng)

Hai phần dưới đây tính Nystrom feature qua CNTK 6 lớp cho toàn bộ dữ liệu
train — nặng hơn nhiều so với GMM/Dictionary Selection/Batch Active Learning
ở trên, nên được đặt sau. Vẫn nhẹ hơn 2 thực nghiệm mạng sâu cuối notebook.


## Tái hiện đúng thực nghiệm gốc trên CIFAR-10 thật (bài báo gốc)

Script `run_algorithm1_variants_paper_cifar10.py` tái hiện đúng thiết lập gốc của paper (ConvNet train trực tiếp trên ảnh -- setting paper *không* dùng ở đây -- đã bị bỏ khỏi notebook này để chỉ dùng 1 pipeline CNTK nhất quán): multiclass logistic regression trên Nystrom feature space (q=2048 chiều) của CNTK 6 lớp + global average pooling, trên CIFAR-10 thật, với đúng các tham số Appendix C (`inner_reg=1e-7`, Adam lr=0.01, warm-start 5e4/1e4 bước, 100 bước conjugate gradient cho implicit gradient).

⚠️ **CẢNH BÁO CHI PHÍ TÍNH TOÁN:** tính CNTK-Nystrom feature cho toàn bộ 45000 ảnh train (kernel 6 lớp conv) rất nặng -- chi phí kernel tỉ lệ với *độ sâu* mạng cho từng cặp (landmark, ảnh), không có "shortcut" như lúc train mạng thật. Có thể mất rất lâu ngay cả trên GPU Kaggle. Feature được cache vào `experiments/cifar10_cntk_features/` sau lần tính đầu tiên và dùng lại cho mọi method/size/seed sau đó -- đây là cách duy nhất để chạy nhiều lần khả thi.

**Khuyến nghị: chạy Ô smoke-test bên dưới trước** (dùng `--nystrom-dim`, `--train-pool-size` nhỏ) để chắc chắn pipeline chạy được trên môi trường Kaggle của bạn (đủ RAM/GPU, jax/neural-tangents hoạt động đúng) trước khi chạy bản đầy đủ đúng tham số paper (có thể mất hàng giờ).

In [ ]:
import subprocess

# Smoke test: dùng --nystrom-dim/--train-pool-size/--val-size/--first-inner-it/--max-inner-it
# nhỏ để kiểm tra pipeline chạy được (jax/neural-tangents, tải CIFAR-10, cache feature...)
# TRƯỚC khi chạy bản đầy đủ đúng tham số paper ở ô dưới. Đây KHÔNG phải tham số mặc định
# của script -- chỉ dùng để test nhanh.
smoke_common = [
    '--nystrom-dim', '128', '--train-pool-size', '2000', '--val-size', '200',
    '--first-inner-it', '200', '--max-inner-it', '50', '--cg-iters', '10',
    '--features-cache-dir', 'experiments/cifar10_cntk_features_smoketest',
    '--output-dir', 'experiments/algo1_paper_cifar10_results_smoketest',
]

subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'full', '--seed', '0'] + smoke_common, check=True)
subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'bico_fwd', '--size-pct', '10', '--seed', '0'] + smoke_common, check=True)
subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'bico_reg', '--size-pct', '10', '--seed', '0',
                '--reg-outer-it', '10'] + smoke_common, check=True)
print("Smoke test OK -- pipeline chạy được. Có thể chạy Ô đầy đủ bên dưới.")

### Chạy đầy đủ đúng tham số paper (nặng -- chỉ chạy sau khi smoke test ở trên đã OK)

Mặc định của script là **đúng giá trị paper** (q=2048, toàn bộ ~45000 ảnh train, 5e4/1e4 bước warm-start, 100 bước CG...) -- không tự động scale nhỏ lại. Sizes theo % của train partition, khớp trục x của Figure 3 (0.5%, 2%, 8%, 32%; 100% = đường "Full Dataset" tham chiếu).

In [ ]:
import subprocess

# 1. Đường tham chiếu "Full Dataset" -- cũng là lần tính CNTK-Nystrom feature đầu tiên
#    (sẽ được cache lại, mọi lệnh sau tái sử dụng, không tính lại kernel).
subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'full', '--seed', '0'], check=True)

sizes_pct = [0.5, 2, 8, 32]

# 2. 4 phương pháp nhị phân
for method in ['uniform', 'bico_fwd', 'bico_fwd25', 'bico_elim', 'bico_exch']:
    for size_pct in sizes_pct:
        subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                        '--method', method, '--size-pct', str(size_pct), '--seed', '0'], check=True)

# 3. bico_reg (weighted, Algorithm 2)
for size_pct in sizes_pct:
    subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                    '--method', 'bico_reg', '--size-pct', str(size_pct), '--seed', '0'], check=True)

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/algorithm1_variants_paper_cifar10_plot.py',
                '--sizes-pct', '0.5,2,8,32', '--seeds', '0'], check=True)

from IPython.display import Image, display
display(Image(filename='experiments/algorithm1_variants_paper_cifar10_accuracy.png'))

## Áp dụng pipeline CNTK-Nystrom lên FashionMNIST

Cùng script `run_algorithm1_variants_paper_cifar10.py`, thêm `--dataset fashionmnist`: dùng lại đúng pipeline CNTK-Nystrom (q=2048) + logistic regression + đúng tham số Appendix C, chỉ đổi dữ liệu đầu vào từ CIFAR-10 sang FashionMNIST. **Đây không phải số liệu paper** -- là kiểm tra xem đúng phương pháp gốc (không đổi gì) có chuyển sang được dataset khác hay không, bổ sung cho (không thay thế) lần chạy CIFAR-10 ở trên. Nên trình bày tách riêng 2 bảng/2 hình trong báo cáo, không gộp chung như thể đang so sánh cùng 1 thực nghiệm trên 2 dataset.

Feature CNTK-Nystrom của Fashion được cache riêng (`experiments/fashionmnist_cntk_features/`), không đụng tới cache của CIFAR-10 -- có thể chạy độc lập, không cần chạy lại Ô CIFAR-10 trước.

In [ ]:
import subprocess

# Smoke test trước, giống Ô CIFAR-10 -- tham số nhỏ, không phải mặc định của script.
smoke_common = [
    '--dataset', 'fashionmnist',
    '--nystrom-dim', '128', '--train-pool-size', '2000', '--val-size', '200',
    '--first-inner-it', '200', '--max-inner-it', '50', '--cg-iters', '10',
    '--features-cache-dir', 'experiments/fashionmnist_cntk_features_smoketest',
    '--output-dir', 'experiments/algo1_paper_fashionmnist_results_smoketest',
]

subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'full', '--seed', '0'] + smoke_common, check=True)
subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'bico_fwd', '--size-pct', '10', '--seed', '0'] + smoke_common, check=True)
subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'bico_reg', '--size-pct', '10', '--seed', '0',
                '--reg-outer-it', '10'] + smoke_common, check=True)
print("Smoke test OK (FashionMNIST) -- có thể chạy Ô đầy đủ bên dưới.")

### Chạy đầy đủ (FashionMNIST, đúng tham số Appendix C -- chỉ đổi dataset)

In [ ]:
import subprocess

subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--dataset', 'fashionmnist', '--method', 'full', '--seed', '0'], check=True)

sizes_pct = [0.5, 2, 8, 32]

for method in ['uniform', 'bico_fwd', 'bico_fwd25', 'bico_elim', 'bico_exch']:
    for size_pct in sizes_pct:
        subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                        '--dataset', 'fashionmnist', '--method', method,
                        '--size-pct', str(size_pct), '--seed', '0'], check=True)

for size_pct in sizes_pct:
    subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                    '--dataset', 'fashionmnist', '--method', 'bico_reg',
                    '--size-pct', str(size_pct), '--seed', '0'], check=True)

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/algorithm1_variants_paper_cifar10_plot.py',
                '--dataset', 'fashionmnist', '--sizes-pct', '0.5,2,8,32', '--seeds', '0'], check=True)

from IPython.display import Image, display
display(Image(filename='experiments/algorithm1_variants_paper_fashionmnist_accuracy.png'))

### Deep network coresets (WideResNet-16-4) -- ĐẶT CUỐI CÙNG

WideResNet-16-4 (2.7 triệu tham số) huấn luyện trực tiếp trên CIFAR-10, coreset chọn qua forward selection theo batch (250 điểm/bước, bắt đầu từ 2500), IHVP qua chuỗi Neumann (T=100). Retrain từ đầu (SGD+momentum, lr=0.1 cosine annealed, weight_decay=5e-4) sau mỗi bước chọn.

⚠️ **RẤT NẶNG -- cố ý đặt cuối cùng.** Theo chính paper, riêng bước tính implicit gradient đã tốn ~2 phút/lần và cần 84 lần để đạt coreset 23500 điểm -- tức khoảng 2-3 tiếng chỉ cho phần chọn, chưa kể retrain WideResNet nhiều chục lần. Bản đầy đủ đúng tham số paper (`--final-size 23500`) **gần chắc chắn vượt quá 12 tiếng/phiên Kaggle**. Ô "chạy đầy đủ" bên dưới dùng `--final-size` rút gọn (không phải số liệu paper) để có cơ hội chạy xong trong 1 phiên -- muốn đúng số liệu paper, sửa `--final-size 23500` và chạy qua nhiều phiên Kaggle nối tiếp (mỗi phiên `git pull` lại rồi chạy tiếp, có thể cần sửa script để checkpoint/resume).

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_neural_network_coresets_paper.py',
                '--dataset', 'cifar10', '--smoke-test',
                '--output-dir', 'experiments/nn_coresets_smoketest'], check=True)
print('Smoke test OK -- co the chay O day du ben duoi (da rut gon final-size).')


**Chạy "đầy đủ" quy mô rút gọn** (`--final-size 3000` thay vì 23500 của paper, các siêu tham số khác giữ đúng paper) -- để có cơ hội chạy xong trong 1 phiên Kaggle. Đây **không phải** số liệu Figure 8/Table 1 của paper, chỉ là đường cong nhỏ hơn cùng phương pháp.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_neural_network_coresets_paper.py',
                '--dataset', 'cifar10', '--final-size', '3000',
                '--checkpoints', '1.0',
                '--output-dir', 'experiments'], check=True)


In [ ]:
from IPython.display import Image, display
display(Image(filename='experiments/neural_network_coresets_cifar10.png'))


### Joint coresets (WideResNet-16-4 + VGG16, transfer sang MobileNetV2) -- ĐẶT CUỐI CÙNG

Coreset chọn đồng thời cho 2 model (luân phiên mỗi bước chọn, λ=1), so sánh với coreset chỉ cho WideResNet khi transfer sang VGG16/MobileNetV2. Tái dùng đúng công thức huấn luyện (SGD+momentum, cosine LR) từ thực nghiệm phía trên.

⚠️ **NẶNG HƠN thực nghiệm phía trên** -- mỗi bước chọn giờ phải huấn luyện lại VÀ tính điểm cho 2 mạng thay vì 1. Cùng lý do, đặt cuối cùng và dùng `--size` rút gọn cho bản "đầy đủ" bên dưới; muốn đúng `--size 23000` của paper (Table 2) cần chạy qua nhiều phiên Kaggle.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_joint_coresets_paper.py',
                '--dataset', 'cifar10', '--smoke-test',
                '--output-dir', 'experiments/joint_coresets_smoketest'], check=True)
print('Smoke test OK -- co the chay O day du ben duoi (da rut gon size).')


**Chạy "đầy đủ" quy mô rút gọn** (`--size 3000` thay vì 23000 của paper). Không phải số liệu Table 2 gốc, chỉ minh hoạ đúng phương pháp.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/run_joint_coresets_paper.py',
                '--dataset', 'cifar10', '--size', '3000',
                '--output-dir', 'experiments'], check=True)


In [ ]:
import pandas as pd
df = pd.read_csv('experiments/joint_coresets_table2_cifar10.csv')
df
